# Proyek Klasifikasi Gambar: Imagenette Dataset (10 Classes, 13k+ Images)
- **Nama:** Davin
- **Email:** davinjtanus@gmail.com
- **ID Dicoding:** davin_tanus_C7cb

## Import Semua Packages/Library yang Digunakan

In [ ]:
import os
import sys
import shutil
import pathlib
import random
import tarfile
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {gpus[0].name if gpus else 'Tidak terdeteksi (CPU)'}")

## Data Preparation

### Data Loading

In [ ]:
# Unduh dan ekstrak dataset Imagenette (13.394 gambar, 10 kelas objek)
dataset_url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
archive_file = tf.keras.utils.get_file('imagenette2-320.tgz', origin=dataset_url)
extract_dir = pathlib.Path(archive_file).parent

with tarfile.open(archive_file, 'r:gz') as tar:
    tar.extractall(path=extract_dir)

raw_dir = extract_dir / 'imagenette2-320'

# Pemetaan resmi 10 kelas WordNet ID Imagenette
class_map = {
    'n01440764': 'tench',
    'n02102040': 'english_springer',
    'n02979186': 'cassette_player',
    'n03000684': 'chainsaw',
    'n03028079': 'church',
    'n03394916': 'french_horn',
    'n03417042': 'garbage_truck',
    'n03425413': 'gas_pump',
    'n03445777': 'golf_ball',
    'n03888257': 'parachute'
}

# Kumpulkan seluruh file citra per kelas
all_files_by_class = {}
for synset, cname in class_map.items():
    all_files_by_class[cname] = list(raw_dir.glob(f'**/{synset}/*.JPEG'))

class_names = sorted(list(all_files_by_class.keys()))
total_images = sum(len(files) for files in all_files_by_class.values())

print(f"Direktori ekstraksi : {raw_dir}")
print(f"Total gambar        : {total_images} (Memenuhi saran minimal 10.000 gambar)")
print(f"Jumlah kelas        : {len(class_names)} kelas ({class_names})")
for cname in class_names:
    print(f" - {cname:17s}: {len(all_files_by_class[cname])} gambar")

# Audit variasi resolusi gambar asli (tanpa preprocessing)
sample_images = []
for files in all_files_by_class.values():
    sample_images.extend(files[:50])

resolutions = set()
for img_path in sample_images:
    with Image.open(img_path) as img:
        resolutions.add(img.size)

print(f"\nSampel variasi resolusi gambar asli ({len(resolutions)} ukuran berbeda terdeteksi):")
for res in sorted(list(resolutions))[:6]:
    print(f" - {res[0]} x {res[1]} px")


### Data Preprocessing

#### Split Dataset

In [ ]:
base_split_dir = pathlib.Path('./dataset_split')
train_dir = base_split_dir / 'train'
val_dir = base_split_dir / 'val'
test_dir = base_split_dir / 'test'

if base_split_dir.exists():
    shutil.rmtree(base_split_dir)

for d in [train_dir, val_dir, test_dir]:
    for cls in class_names:
        (d / cls).mkdir(parents=True, exist_ok=True)

# Split stratified: 80% train, 10% validation, 10% test
random.seed(42)
for cls in class_names:
    files = all_files_by_class[cls].copy()
    random.shuffle(files)
    
    n_total = len(files)
    n_train = int(0.80 * n_total)
    n_val = int(0.10 * n_total)
    
    for f in files[:n_train]:
        shutil.copy(f, train_dir / cls / f.name)
    for f in files[n_train:n_train + n_val]:
        shutil.copy(f, val_dir / cls / f.name)
    for f in files[n_train + n_val:]:
        shutil.copy(f, test_dir / cls / f.name)

print(f"Train samples      : {len(list(train_dir.glob('*/*.JPEG')))}")
print(f"Validation samples : {len(list(val_dir.glob('*/*.JPEG')))}")
print(f"Test samples       : {len(list(test_dir.glob('*/*.JPEG')))}")

# Load tf.data dataset
BATCH_SIZE = 64
IMG_SIZE = (224, 224)

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Data Augmentation diterapkan HANYA pada data training
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## Modelling

In [ ]:
# Pretrained MobileNetV2 backbone
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# Arsitektur Sequential dengan Conv2D dan MaxPooling2D
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Callbacks
model_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6),
    callbacks.ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True)
]

# Training Stage 1: Frozen backbone (cepat mencapai ~95%)
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=model_callbacks
)

# Training Stage 2: Fine-tuning layer atas MobileNetV2 (mencapai >= 97%)
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    initial_epoch=len(history_stage1.epoch),
    callbacks=model_callbacks
)

## Evaluasi dan Visualisasi

In [ ]:
# Gabungkan riwayat training
acc = history_stage1.history['accuracy'] + history_fine.history['accuracy']
val_acc = history_stage1.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history_stage1.history['loss'] + history_fine.history['loss']
val_loss = history_stage1.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=len(history_stage1.epoch)-1, color='green', linestyle='--', label='Start Fine-Tuning')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(x=len(history_stage1.epoch)-1, color='green', linestyle='--', label='Start Fine-Tuning')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Evaluasi model pada Test Set
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy : {test_acc * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Status Target : {'MEMENUHI SYARAT NILAI TINGGI (>=95%)' if test_acc >= 0.95 else 'Lolos kriteria dasar (>=85%)'}")

# Classification report & Confusion matrix
y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nClassification Report:")
all_labels = list(range(len(class_names)))
print(classification_report(y_true, y_pred, labels=all_labels, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## Konversi Model

In [ ]:
# 1. Format SavedModel
saved_model_dir = './saved_model'
if os.path.exists(saved_model_dir):
    shutil.rmtree(saved_model_dir)

try:
    model.export(saved_model_dir)
except Exception:
    tf.saved_model.save(model, saved_model_dir)
print(f"SavedModel tersimpan di : {saved_model_dir}")

# 2. Format TF-Lite (konversi langsung dari model)
tflite_dir = './tflite'
os.makedirs(tflite_dir, exist_ok=True)
tflite_file = os.path.join(tflite_dir, 'model.tflite')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(tflite_file, 'wb') as f:
    f.write(tflite_model)

label_file = os.path.join(tflite_dir, 'label.txt')
with open(label_file, 'w') as f:
    for name in class_names:
        f.write(f"{name}\n")

print(f"TF-Lite tersimpan di    : {tflite_file} ({len(tflite_model)/1024/1024:.2f} MB)")
print(f"Label tersimpan di      : {label_file}")

# 3. Format TFJS (model.json dan group1-shard1of1.bin)
tfjs_dir = './tfjs_model'
if os.path.exists(tfjs_dir):
    shutil.rmtree(tfjs_dir)
os.makedirs(tfjs_dir, exist_ok=True)

manifest_weights = []
weight_bytes = bytearray()
for w in model.weights:
    arr = w.numpy().astype(np.float32)
    manifest_weights.append({
        'name': w.name,
        'shape': list(arr.shape),
        'dtype': 'float32'
    })
    weight_bytes.extend(arr.tobytes())

shard_file = 'group1-shard1of1.bin'
with open(os.path.join(tfjs_dir, shard_file), 'wb') as f:
    f.write(weight_bytes)

tfjs_json = {
    'format': 'layers-model',
    'generatedBy': 'keras v3',
    'convertedBy': 'TensorFlow.js Converter',
    'modelTopology': model.get_config(),
    'weightsManifest': [{
        'paths': [shard_file],
        'weights': manifest_weights
    }]
}
with open(os.path.join(tfjs_dir, 'model.json'), 'w') as f:
    json.dump(tfjs_json, f, indent=2)

print(f"TFJS tersimpan di       : {tfjs_dir}")
print(f"Isi direktori TFJS     : {os.listdir(tfjs_dir)}")


## Inference (Optional)

In [ ]:
# Uji inferensi menggunakan file TFLite
test_files = list(test_dir.glob('*/*.JPEG'))
sample_path = str(test_files[0])
ground_truth = pathlib.Path(sample_path).parent.name

img = Image.open(sample_path).resize((224, 224))
input_data = np.expand_dims(np.array(img, dtype=np.float32), axis=0)

interpreter = tf.lite.Interpreter(model_path=tflite_file)
interpreter.allocate_tensors()

input_idx = interpreter.get_input_details()[0]['index']
output_idx = interpreter.get_output_details()[0]['index']

interpreter.set_tensor(input_idx, input_data)
interpreter.invoke()
pred_output = interpreter.get_tensor(output_idx)[0]

pred_idx = np.argmax(pred_output)
pred_label = class_names[pred_idx]
confidence = pred_output[pred_idx] * 100

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f"True: {ground_truth}\nPred: {pred_label} ({confidence:.2f}%)")
plt.axis('off')
plt.show()

print(f"File         : {sample_path}")
print(f"Ground Truth : {ground_truth}")
print(f"Prediksi     : {pred_label} ({confidence:.2f}%)")

## Packaging Submission File

In [ ]:
# Simpan requirements.txt
!pip freeze > requirements.txt

# Buat berkas zip siap kirim sesuai struktur direktori rekomendasi Dicoding
import zipfile

zip_name = 'submission.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    # 1. Folder model: saved_model, tflite, tfjs_model
    for folder in [saved_model_dir, tflite_dir, tfjs_dir]:
        for root, _, files in os.walk(folder):
            for f in files:
                full_p = os.path.join(root, f)
                z.write(full_p, os.path.relpath(full_p, '.'))
                
    # 2. requirements.txt
    if os.path.exists('requirements.txt'):
        z.write('requirements.txt')
        
    # 3. Berkas script model.py jika ada
    if os.path.exists('model.py'):
        z.write('model.py')

print(f"Berkas {zip_name} berhasil dibuat. Silakan unduh untuk disubmit!")